# TimbreTrack — a simple example

Extract psychoacoustic timbre features from a `.wav` file with
`comsar.TimbreTrack`, then visualise them: the waveform is drawn in grey with the
feature tracks overlaid in colour, together with a **play button** and a cursor
that follows the audio.

> Requires `bader-comsar` (see the [main README](../README.md)) plus
> `soundfile`, and a browser that can play WAV audio.

In [45]:
from comsar import TimbreTrack
from comsar.viz import timbre_player
from apollon.signal import container

## Analysis parameters

`TimbreTrack` cuts the recording into short **analysis windows** and computes
every timbre feature once per window. The window setup is controlled by an
`StftParams` object:

| Parameter | Meaning |
|---|---|
| `fps` | Sample rate in Hz. **Must match the audio file**, otherwise `extract` raises an error. |
| `window` | Window function applied to each segment before the FFT (e.g. `'hamming'`). |
| `n_fft` | FFT length; `None` uses the window size. |
| `n_perseg` | Window length in **samples**. Longer windows → finer frequency resolution, coarser time resolution. |
| `n_overlap` | Overlap between consecutive windows in **samples** (must be smaller than `n_perseg`). Overlapping windows give a denser, smoother feature track. |
| `extend`, `pad` | Add half a window of padding at the start/end so the first and last frames are properly centred. |

The **hop size** (time step between two feature values) is
`n_perseg - n_overlap` samples. Both values are counted in **whole samples**
(integers) — that is why the overlap is specified as a fraction below and then
converted with `int(...)`. More overlap means more frames and therefore a
longer computation (the correlation dimension dominates the runtime): at
44.1 kHz with 16 384-sample windows, 50 % overlap gives one value every
≈ 0.19 s, 80 % overlap one every ≈ 0.07 s.

In [46]:
# --- Analysis parameters ------------------------------------------------------
SAMPLE_RATE      = 44100    # Hz - must match the audio file
WINDOW_SIZE      = 2**14    # samples per analysis window (16384 ~ 0.37 s)
OVERLAP_FRACTION = 0.8      # fraction of the window that overlaps (0 ... <1)

# Windowing works in whole samples, so the overlap is rounded to an integer.
OVERLAP = int(WINDOW_SIZE * OVERLAP_FRACTION)

stft_params = container.StftParams(
    fps=SAMPLE_RATE,
    window='hamming',
    n_fft=None,
    n_perseg=WINDOW_SIZE,
    n_overlap=OVERLAP,
    extend=True,
    pad=True,
)

# Initialise the timbre track with overlapping analysis windows
tt = TimbreTrack(stft_params=stft_params)
print(f"hop = {WINDOW_SIZE - OVERLAP} samples "
      f"= {(WINDOW_SIZE - OVERLAP) / SAMPLE_RATE:.3f} s per feature value")

hop = 3277 samples = 0.074 s per feature value


In [47]:
# Extract psychoacoustic features from the recording. The path is relative, so
# the .wav file has to sit next to this notebook. `extract` returns a
# TrackResult; `.features` is a pandas DataFrame with one row per analysis frame.
WAV = "CHI-109_Luguhu_Hulusheng.wav"
result = tt.extract(WAV)
features = result.features
features.head()

SpectralCentroid ... 0.185 s.
SpectralSpread ... 0.4487 s.
SpectralFlux ... 0.05049 s.
Roughness ... 0.2808 s.
Sharpness ... 0.005975 s.
SPL ... 0.05293 s.
CorrelationDimension ... 68.38 s.


,SpectralCentroid,SpectralSpread,SpectralFlux,Roughness,Sharpness,SPL,CorrelationDimension
0,1154.187760,543.670386,0.892687,0.181740,0.001323,75.590277,0.000000
1,1323.754937,508.320381,0.472689,0.130023,0.001374,76.224103,0.000000
2,1383.435640,488.466104,0.109744,0.085215,0.001030,77.774499,0.000025
3,1374.952369,518.452236,0.209983,0.097115,0.000805,78.677057,2.654702
4,1424.843881,535.576200,0.276908,0.084474,0.000869,78.977565,3.079535


## Save the results as CSV

The extracted features are saved right away, so the analysis is not lost if the
notebook is closed. A `time_s` column (frame start = frame index × hop size) is
added, and the file is written next to the audio file, e.g.
`CHI-109_Luguhu_Hulusheng_timbre.csv`.

**Excel note.** German/European Excel uses the comma as *decimal* sign and
therefore expects **semicolons** between columns — a standard comma-separated
CSV shows up as a single column there. With `EXCEL_STYLE = True` (default) the
file is written with `;` separators and decimal commas so it opens cleanly in
European Excel. Set it to `False` for a standard comma CSV (pandas, R, US
Excel). To read the Excel-style file back with pandas:
`pd.read_csv(CSV_FILE, sep=";", decimal=",")`.

(For JSON or pickle output see `result.to_json` / `result.to_pickle`.)

In [48]:
# Save the feature table as CSV, with a time axis in seconds
CSV_FILE = WAV.rsplit(".", 1)[0] + "_timbre.csv"

# German/European Excel expects ';' as column separator and ',' as decimal
# sign. Set EXCEL_STYLE = False for a standard comma-separated CSV.
EXCEL_STYLE = True

csv_data = features.copy()
csv_data.insert(0, "time_s",
                csv_data.index * (WINDOW_SIZE - OVERLAP) / SAMPLE_RATE)
if EXCEL_STYLE:
    csv_data.to_csv(CSV_FILE, index=False, sep=";", decimal=",")
else:
    csv_data.to_csv(CSV_FILE, index=False)
print(f"saved {len(csv_data)} frames x {len(features.columns)} features to {CSV_FILE}")

saved 241 frames x 7 features to CHI-109_Luguhu_Hulusheng_timbre.csv


## Visualisation

`comsar.viz.timbre_player` draws the waveform in light grey with each feature
normalised to `[0, 1]` and overlaid in its own colour. Press **Play** to listen
— a vertical cursor follows the playback position, and clicking the plot seeks
to that point.

To keep the plot readable, only the **first two features are shown initially**;
**click a legend entry** to show or hide a feature (hidden entries are grey).
Use `visible=None` to start with all features shown, e.g.
`timbre_player(WAV, features, visible=None)`.

The player is a self-contained HTML/JavaScript widget (audio embedded as a data
URI), so it also works after the notebook is exported to HTML.

In [49]:
timbre_player(WAV, features)